<a href="https://colab.research.google.com/github/lamia-khd/react-router/blob/main/calcule2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import time # Imports the built-in module used to measure elapsed time
import numpy as np # Imports the NumPy library for fast numerical operations
from concurrent.futures import ThreadPoolExecutor # Imports the tool to manage multi-threading

def read_config(): # Defines the function that simulates reading the initial settings
    """Settings file: read first, by one worker only (2 s).""" # Documentation string for the setup function
    time.sleep(2) # Pauses the program for 2 seconds to simulate sequential read time

def download(seconds): # Defines the function simulating the file download process
    """Download one shop's sales file (it takes some seconds).""" # Documentation string for the task
    time.sleep(seconds) # Pauses for the amount of seconds passed as an argument

files = [1, 1, 1, 1, 1, 1, 1, 1] # List representing today's 8 files, each taking 1 second to download
tomorrow = [4, 1, 1, 1, 1, 1, 1, 2] # List representing tomorrow's files with varying download times
rng = np.random.default_rng(0) # Initializes the random number generator with a fixed seed
amounts = rng.integers(100, 5000, 2_000_000) # Generates an array of 2 million random sales figures

# Q1
t0 = time.perf_counter() # Starts the stopwatch for the single-worker execution
read_config() # Executes the mandatory 2-second sequential configuration read
for f in files: # Loops through every file in today's list one by one
    download(f) # Downloads the current file in the sequence
T1 = time.perf_counter() - t0 # Calculates the total execution time for 1 worker
print(f"Q1 1 worker: {T1:.1f} s") # Prints the resulting time

# Q2
t0 = time.perf_counter() # Resets and starts the stopwatch for the parallel execution
read_config() # Executes the sequential configuration step again
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 independent workers
    list(team.map(download, files)) # Assigns the download tasks for the 8 files to the team
Tp = time.perf_counter() - t0 # Calculates the total execution time with 4 workers
print(f"Q2 4 workers: {Tp:.1f} s, speed-up {T1 / Tp:.1f}") # Prints the time and the calculated speed-up

# Q3
part_alone = 2.0 # Defines the time taken by the indivisible sequential portion
print(f"Q3 alone: {part_alone:.1f} s, best possible: {T1 / part_alone:.1f}") # Prints max theoretical speed-up

# Q4
t_loop = time.perf_counter() # Starts the timer for the standard Python loop addition
total_loop = sum(amounts) # Calculates the sum using Python's built-in sum function
loop_time = time.perf_counter() - t_loop # Records the time taken by the standard loop

t_numpy = time.perf_counter() # Starts the timer for the optimized NumPy addition
total_numpy = amounts.sum() # Calculates the sum using NumPy's highly optimized SIMD method
numpy_time = time.perf_counter() - t_numpy # Records the time taken by NumPy
print(f"Q4 loop {loop_time:.4f} s, numpy {numpy_time:.4f} s, same total: {total_loop == total_numpy}") # Prints comparison

# Q5
t0 = time.perf_counter() # Starts the stopwatch for tomorrow's uneven schedule
read_config() # Executes the required 2-second setup
with ThreadPoolExecutor(max_workers=4) as team: # Creates a team of 4 workers for the new workload
    list(team.map(download, tomorrow)) # Executes tomorrow's varied tasks in parallel
print(f"Q5 tomorrow, 4 workers: {time.perf_counter() - t0:.1f} s") # Prints the actual completion time

Q1 1 worker: 10.0 s
Q2 4 workers: 4.0 s, speed-up 2.5
Q3 alone: 2.0 s, best possible: 5.0
Q4 loop 0.1574 s, numpy 0.0016 s, same total: True
Q5 tomorrow, 4 workers: 6.0 s


In [ ]:

T1 = 100      # Total training time on one core, in minutes
alone = 5     # Time that cannot be parallelized
T8 = 16.9     # Measured time on the 8-core server
T64 = 6.5     # Measured time on the 64-core server

# Q1: Calculate the speed-up on the 8-core server
speedup8 = T1 / T8
print(f"Q1 8 cores: x{speedup8:.2f}")

# Q2: Calculate the speed-up on the 64-core server
speedup64 = T1 / T64
print(f"Q2 64 cores: x{speedup64:.2f}")

# Q3: Compare the speed-up of the two servers
comparison = T8 / T64
print(f"Q3 big / small: x{comparison:.2f}")

# Q4: Calculate the best possible speed-up and training time
# The 5-minute part cannot be parallelized
best_time = alone
best_speedup = T1 / best_time
print(f"Q4 best possible: x{best_speedup:.2f}, {best_time:.2f} min")

# Q5: Calculate the new best speed-up if the non-parallelizable
# part is reduced to 1 minute
new_alone = 1
new_best_speedup = T1 / new_alone
print(f"Q5 alone = 1 min: best x{new_best_speedup:.2f}")

Q1 8 cores: x5.92
Q2 64 cores: x15.38
Q3 big / small: x2.60
Q4 best possible: x20.00, 5.00 min
Q5 alone = 1 min: best x100.00


### Explanation of the Notebook Results

#### Section 1: Parallelization and SIMD
* **Q1: 1 Worker (Sequential Execution)**
  * **Result:** `10.0 s`
  * **Explanation:** Reading the configurations takes a mandatory 2 seconds of sequential time. Following that, downloading 8 files (each taking 1 second) sequentially adds 8 seconds. In total, the single-worker pipeline takes $2 + 8 = 10$ seconds.

* **Q2: 4 Workers (Parallel Execution)**
  * **Result:** `4.0 s` (Speed-up: `2.5x`)
  * **Explanation:** The 2-second sequential configuration read cannot be parallelized. However, the 8 downloading tasks (1 second each) are split across 4 workers. Each worker processes 2 files, taking 2 seconds in parallel. The total runtime is $2\text{ s (config)} + 2\text{ s (parallel downloads)} = 4\text{ s}$.

* **Q3: Amdahl's Law Limitation**
  * **Result:** Best possible: `5.0x`
  * **Explanation:** According to Amdahl's Law, because the 2-second configuration part must run sequentially, the total execution time can never drop below 2 seconds, no matter how many parallel workers are added. Thus, the theoretical maximum speed-up is $10\text{ s (total)} / 2\text{ s (serial)} = 5.0\text{x}$.

* **Q4: Built-in Loop vs. NumPy SIMD**
  * **Result:** Python loop `~0.1574 s` vs. NumPy `~0.0016 s`
  * **Explanation:** NumPy operations utilize vectorization and Single Instruction Multiple Data (SIMD) processor instructions compiled in C. This allows NumPy to sum millions of values almost instantly, outperforming standard Python loops which suffer from interpreter overhead.

* **Q5: Tomorrow's Load Balancing**
  * **Result:** `6.0 s`
  * **Explanation:** Tomorrow's file downloads take variable times: `[4, 1, 1, 1, 1, 1, 1, 2]`. Since one file takes 4 seconds, even with 4 workers, the download stage cannot finish in less than 4 seconds. Consequently, the total time becomes $2\text{ s (config)} + 4\text{ s (longest file)} = 6.0\text{ s}$.

---

#### Section 2: Amdahl's Law and Scaling
* **Q1: 8 Cores Speed-up**
  * **Result:** `x5.92`
  * **Explanation:** Training sequentially takes 100 minutes. On 8 cores, it takes 16.9 minutes. The resulting speedup is $100 / 16.9 \approx 5.92\text{x}$.

* **Q2: 64 Cores Speed-up**
  * **Result:** `x15.38`
  * **Explanation:** On 64 cores, the execution time drops to 6.5 minutes. The resulting speedup is $100 / 6.5 \approx 15.38\text{x}$. This highlights diminishing returns as we add more cores because of the serial portion.

* **Q3: Relative Performance (64 Cores vs. 8 Cores)**
  * **Result:** `x2.60`
  * **Explanation:** Despite having 8 times more processors ($64 / 8$), the 64-core server is only 2.6 times faster than the 8-core server due to the bottleneck of the serial portion.

* **Q4: Theoretical Limits**
  * **Result:** Max Speed-up `x20.00`, minimum training time `5.00 min`
  * **Explanation:** Because 5 minutes of the training process cannot be parallelized, the execution time can never go below 5 minutes. The absolute maximum speedup limit is $100 / 5 = 20.00\text{x}$.

* **Q5: Optimization Impact**
  * **Result:** Max Speed-up `x100.00`
  * **Explanation:** If we optimize the non-parallelizable portion and reduce it from 5 minutes to 1 minute, the theoretical maximum speed-up jumps significantly to $100 / 1 = 100.00\text{x}$.